# The Divided Danish Home — Analysis Notebook
**Structure:** All data loading and preparation happens in Section 1. Each plot cell below is fully self-contained and can be deleted without affecting others.

---
## 1. Imports & Configuration

In [23]:
import os
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

os.makedirs('Figures', exist_ok=True)

---
## 2. Load & Clean Data
All name standardisation happens here. `df_mun` and `df_reg` produced below are the single source of truth for every plot.

In [24]:
# ── Municipality name map (CSV names → GeoJSON names) ──────────────────────
NAME_MAP = {
    'Copenhagen':  'Københavns',
    'Vesthimmerland': 'Vesthimmerlands',
    'Bornholm':    'Bornholms Regionskommune',
}

# ── Load municipality data ──────────────────────────────────────────────────
df_mun = pd.read_csv('FinalData/all_data_mun.csv')
df_mun['Municipality'] = df_mun['Municipality'].replace(NAME_MAP)
df_mun['affordability'] = df_mun['income'] / df_mun['sqm_price_mean']
df_mun['sales_per_1000'] = df_mun['no_sales'] / df_mun['population'] * 1000
df_mun = df_mun.sort_values(['Municipality', 'year']).reset_index(drop=True)

# ── Urban / rural classification (median population in 2024) ───────────────
pop_2024 = df_mun[df_mun['year'] == 2024][['Municipality', 'population']]
threshold = pop_2024['population'].median()
urban_set = set(pop_2024[pop_2024['population'] >= threshold]['Municipality'])
df_mun['urban_rural'] = df_mun['Municipality'].apply(
    lambda x: 'Urban' if x in urban_set else 'Rural'
)

# ── Load regional data ──────────────────────────────────────────────────────
df_reg = pd.read_csv('FinalData/all_data_reg.csv')
df_reg['affordability'] = df_reg['income'] / df_reg['sqm_price_mean']

print('df_mun:', df_mun.shape, '| years:', df_mun['year'].min(), '–', df_mun['year'].max())
print('df_reg:', df_reg.shape)
df_mun.head(3)

df_mun: (1666, 19) | years: 2008 – 2024
df_reg: (85, 16)


,Municipality,Region,year,purchase_price_mean,purchase_price_median,sqm_price_mean,sqm_price_median,sqm_mean,no_rooms_mean,%_change_between_offer_and_purchase_mean,nom_interest_rate%_mean,dk_ann_infl_rate%_mean,yield_on_mortgage_credit_bonds%_mean,no_sales,income,population,affordability,sales_per_1000,urban_rural
0,Aabenraa,Southern Denmark,2008,1.631118e+06,1065000.0,12842.465,8209.458984,142.54176,4.691649,-1.708779,4.046039,3.42,6.52,467,276912.666667,60316.75,21.562268,7.742460,Urban
1,Aabenraa,Southern Denmark,2009,1.535254e+06,1025000.0,11637.358,8295.454102,147.15804,4.762943,-2.779292,1.916621,1.30,5.50,367,278411.000000,60237.00,23.923901,6.092601,Urban
2,Aabenraa,Southern Denmark,2010,1.577195e+06,1140000.0,12633.355,8218.090332,143.67287,4.609665,-3.046469,0.795074,2.31,4.53,538,301608.000000,59946.00,23.873943,8.974744,Urban


---
## 3. Load GeoJSON & Build Derived DataFrames
All cross-year comparison frames (migration, growth, income gap) are built once here.

In [25]:
# ── GeoJSON ────────────────────────────────────────────────────────────────
with open('GeojsonFiles/denmark-municipalities.geojson', 'r') as f:
    geojson_mun = json.load(f)

# Standardise GeoJSON names to match df_mun
GEOJSON_FIX = {
    'Københavns':             'Københavns',      # already correct
    'Vesthimmerlands':        'Vesthimmerlands',  # already correct
    'Bornholms Regionskommune': 'Bornholms Regionskommune',
}
for feat in geojson_mun['features']:
    clean = feat['properties']['name'].replace(' Kommune', '')
    feat['properties']['name_clean'] = clean  # no extra remapping needed now

geojson_names = {f['properties']['name_clean'] for f in geojson_mun['features']}

# ── Region GeoJSON ─────────────────────────────────────────────────────────
with open('GeojsonFiles/denmark_regions.geojson', 'r') as f:
    geojson_reg = json.load(f)

REGION_NAME_MAP = {
    'Region Nordjylland': 'North Jutland',
    'Region Midtjylland': 'Central Jutland',
    'Region Syddanmark':  'Southern Denmark',
    'Region Hovedstaden': 'Capital Region',
    'Region Sjælland':    'Zealand Region',
}
for feat in geojson_reg['features']:
    feat['properties']['navn_en'] = REGION_NAME_MAP.get(feat['properties']['navn'], feat['properties']['navn'])

print('GeoJSON municipalities:', len(geojson_names))
missing = geojson_names - set(df_mun['Municipality'].unique())
print('In GeoJSON but not in df_mun:', missing)

GeoJSON municipalities: 98
In GeoJSON but not in df_mun: set()


In [26]:
df_08 = df_mun[df_mun['year'] == 2008][['Municipality', 'Region', 'population', 'affordability']].rename(
    columns={'population': 'pop_2008', 'affordability': 'aff_2008'})
df_24 = df_mun[df_mun['year'] == 2024][['Municipality', 'population', 'affordability']].rename(
    columns={'population': 'pop_2024', 'affordability': 'aff_2024'})

df_migration = df_08.merge(df_24, on='Municipality')

# DEBUG: see what's NaN before dropping
print(df_migration[df_migration['Municipality'].str.contains('benhavn|esthimmer', case=False)])

# Drop only rows missing the columns we actually calculate from
df_migration = df_migration.dropna(subset=['pop_2008', 'pop_2024', 'aff_2008', 'aff_2024'])

df_migration['pop_growth_pct'] = (
    (df_migration['pop_2024'] - df_migration['pop_2008']) / df_migration['pop_2008'] * 100
)
df_migration['aff_change'] = df_migration['aff_2024'] - df_migration['aff_2008']

# ── df_growth: income & sqm price growth 2008 → 2024 per municipality ──────
growth = df_mun.groupby('Municipality').agg(
    income_start=('income', 'first'),
    income_end=('income', 'last'),
    sqm_start=('sqm_price_mean', 'first'),
    sqm_end=('sqm_price_mean', 'last'),
).reset_index()
growth['income_growth_pct'] = (
    (growth['income_end'] - growth['income_start']) / growth['income_start'].replace(0, pd.NA) * 100
)
growth['sqm_price_growth_pct'] = (
    (growth['sqm_end'] - growth['sqm_start']) / growth['sqm_start'].replace(0, pd.NA) * 100
)
df_growth = growth[['Municipality', 'income_growth_pct', 'sqm_price_growth_pct']]

# ── df_map: align everything to GeoJSON index ──────────────────────────────
df_map = pd.DataFrame({'Municipality': sorted(geojson_names)})
df_map = df_map.merge(df_migration[['Municipality', 'pop_growth_pct', 'aff_change']], on='Municipality', how='left')
df_map = df_map.merge(df_growth, on='Municipality', how='left')
df_map[['pop_growth_pct', 'aff_change', 'income_growth_pct', 'sqm_price_growth_pct']] = \
    df_map[['pop_growth_pct', 'aff_change', 'income_growth_pct', 'sqm_price_growth_pct']].fillna(0)

# ── df_income_gap: income needed to maintain 2008 affordability ────────────
df_base_aff = df_mun[df_mun['year'] == 2008][['Municipality', 'affordability', 'income']].rename(
    columns={'affordability': 'aff_2008', 'income': 'income_2008'})
df_mun_gap = df_mun.merge(df_base_aff, on='Municipality')
df_mun_gap['required_income'] = df_mun_gap['aff_2008'] * df_mun_gap['sqm_price_mean']
df_mun_gap['income_gap'] = df_mun_gap['required_income'] - df_mun_gap['income']

df_income_gap = (
    df_mun_gap.dropna(subset=['income_gap', 'population'])
    .groupby(['year', 'urban_rural'])
    .apply(lambda g: (g['income_gap'] * g['population']).sum() / g['population'].sum())
    .reset_index(name='avg_income_gap')
)

# ── df_weighted: population-weighted affordability urban vs rural ───────────
df_weighted = (
    df_mun.dropna(subset=['population', 'affordability'])
    .groupby(['year', 'urban_rural'])
    .apply(lambda g: (g['affordability'] * g['population']).sum() / g['population'].sum())
    .reset_index(name='weighted_affordability')
)
df_gap_pivot = df_weighted.pivot(index='year', columns='urban_rural', values='weighted_affordability').reset_index()
df_gap_pivot['gap'] = df_gap_pivot['Rural'] - df_gap_pivot['Urban']

# ── national aggregates ────────────────────────────────────────────────────
df_national = (
    df_mun.groupby('year')
    .apply(lambda g: pd.Series({
        'affordability': (g['income'] / g['sqm_price_mean'] * g['population']).sum() / g['population'].sum(),
        'total_sales':   g['no_sales'].sum(),
    }))
    .reset_index()
)

print('All derived frames ready.')
print('df_migration:', df_migration.shape)
print('df_map:', df_map.shape, '| zeros in pop_growth_pct:',
      (df_map['pop_growth_pct'] == 0).sum())

       Municipality          Region  pop_2008   aff_2008  pop_2024   aff_2024
50       Københavns  Capital Region  512558.5   9.445433  662770.5   9.505779
94  Vesthimmerlands   North Jutland   38421.5  26.528877   35920.5  34.952452
All derived frames ready.
df_migration: (98, 8)
df_map: (98, 5) | zeros in pop_growth_pct: 0


---
## 4. Plots
Each cell below is independent. Delete any cell without affecting the others.

### Fig 1 — National affordability trend

In [27]:
df_nat_reg = df_reg.groupby('year')['affordability'].mean().reset_index()

fig = px.line(
    df_nat_reg, x='year', y='affordability',
    labels={'affordability': 'Affordability Index (Income / sqm price)', 'year': 'Year'},
    markers=True,
)
fig.update_layout(hovermode='x unified')
fig.write_html('Figures/national_affordability_trend.html')
fig.show()

### Fig 2a — Affordability vs interest rates

In [28]:
df_nat_full = df_reg.groupby('year').agg(
    affordability=('affordability', 'mean'),
    interest_rate=('nom_interest_rate%_mean', 'mean'),
).reset_index()

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(
    go.Scatter(x=df_nat_full['year'], y=df_nat_full['affordability'],
               name='Affordability Index', line=dict(color='#2ecc71', width=2.5), mode='lines+markers'),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=df_nat_full['year'], y=df_nat_full['interest_rate'],
               name='Nominal Interest Rate (%)', line=dict(color='#e74c3c', width=2.5, dash='dash'),
               mode='lines+markers'),
    secondary_y=True,
)
fig.update_layout(hovermode='x unified')
fig.update_yaxes(title_text='Affordability Index', secondary_y=False)
fig.update_yaxes(title_text='Nominal Interest Rate (%)', secondary_y=True)
fig.write_html('Figures/affordability_vs_interest.html')
fig.show()

### Fig 2b — Transaction volume vs affordability

In [29]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=df_national['year'], y=df_national['total_sales'],
    name='Annual transactions', yaxis='y2',
    marker_color='rgba(120,160,180,0.35)', marker_line_width=0,
))
fig.add_trace(go.Scatter(
    x=df_national['year'], y=df_national['affordability'].round(2),
    name='Affordability index', yaxis='y1',
    mode='lines+markers',
    line=dict(color='#e07832', width=2.5),
    marker=dict(size=6, color='#e07832'),
))

for x0, x1, label in [(2008, 2009.5, 'Financial crisis'), (2022, 2023, 'Rate shock')]:
    fig.add_vrect(x0=x0, x1=x1, fillcolor='rgba(180,60,20,0.07)', line_width=0,
                  annotation_text=label, annotation_position='top left',
                  annotation_font_size=11, annotation_font_color='#888')

fig.update_layout(
    xaxis=dict(tickmode='linear', dtick=1, title=''),
    yaxis=dict(title='Affordability index', side='left',
               showgrid=True, gridcolor='rgba(0,0,0,0.06)'),
    yaxis2=dict(title='Annual transactions', side='right', overlaying='y', showgrid=False),
    legend=dict(orientation='h', y=-0.15),
    plot_bgcolor='white', paper_bgcolor='white',
    hovermode='x unified', bargap=0.15,
    margin=dict(t=60, b=60),
)
fig.write_html('Figures/volume_vs_affordability.html')
fig.show()

### Fig 3 — Regional affordability lines

In [30]:
fig = px.line(
    df_reg, x='year', y='affordability', color='Region',
    labels={'affordability': 'Affordability Index', 'year': 'Year'},
    markers=True,
)
fig.update_layout(hovermode='x unified')
fig.write_html('Figures/regional_affordability_lines.html')
fig.show()

### Fig 4 — Urban vs rural weighted affordability gap

In [31]:
fig = make_subplots(specs=[[{'secondary_y': True}]])

for label, color in [('Urban', '#e74c3c'), ('Rural', '#2ecc71')]:
    sub = df_weighted[df_weighted['urban_rural'] == label]
    fig.add_trace(
        go.Scatter(x=sub['year'], y=sub['weighted_affordability'],
                   name=label, line=dict(color=color, width=2.5), mode='lines+markers'),
        secondary_y=False,
    )

fig.add_trace(
    go.Bar(x=df_gap_pivot['year'], y=df_gap_pivot['gap'],
           name='Rural–Urban Gap', marker_color='rgba(52,152,219,0.4)'),
    secondary_y=True,
)
fig.update_layout(hovermode='x unified', bargap=0.3)
fig.update_yaxes(title_text='Weighted Affordability Index', secondary_y=False)
fig.update_yaxes(title_text='Affordability Gap (Rural − Urban)', secondary_y=True)
fig.write_html('Figures/weighted_affordability_gap.html')
fig.show()

### Fig 5 — Income gap to maintain 2008 affordability

In [32]:
fig = px.area(
    df_income_gap, x='year', y='avg_income_gap', color='urban_rural',
    labels={'avg_income_gap': 'Income Gap (DKK/year)', 'year': 'Year', 'urban_rural': ''},
    color_discrete_map={'Urban': '#e74c3c', 'Rural': '#2ecc71'},
)
fig.add_hline(y=0, line_dash='dash', line_color='black', annotation_text='No gap (2008 baseline)')
fig.update_layout(hovermode='x unified')
fig.write_html('Figures/income_gap.html')
fig.show()

### Fig 6a — Population growth vs 2008 affordability (scatter)

In [33]:
fig = px.scatter(
    df_migration,
    x='aff_2008', y='pop_growth_pct',
    color='Region', size='pop_2024',
    hover_name='Municipality', size_max=40,
    labels={
        'aff_2008': 'Affordability in 2008 (baseline)',
        'pop_growth_pct': 'Population Growth 2008–2024 (%)',
    },
    hover_data={'aff_change': ':.2f', 'pop_2024': ':,.0f'},
)
# Trendline
z = np.polyfit(df_migration['aff_2008'], df_migration['pop_growth_pct'], 1)
x_line = np.linspace(df_migration['aff_2008'].min(), df_migration['aff_2008'].max(), 100)
fig.add_trace(go.Scatter(x=x_line, y=np.poly1d(z)(x_line), mode='lines',
                          name='Trend', line=dict(color='black', dash='dash', width=1.5)))
fig.update_layout(hovermode='closest')
fig.write_html('Figures/population_vs_affordability.html')
fig.show()

### Fig 6b — Population growth by affordability change group (box plot)

In [34]:
df_box = df_migration.copy()
df_box['aff_change_group'] = pd.cut(
    df_box['aff_change'],
    bins=[-999, -5, 0, 5, 999],
    labels=['Large decline (< -5)', 'Small decline (0 to -5)', 'Small gain (0 to 5)', 'Large gain (> 5)']
)

fig = px.box(
    df_box, x='aff_change_group', y='pop_growth_pct',
    color='aff_change_group', hover_name='Municipality', points='all',
    labels={
        'aff_change_group': 'Change in Affordability (2008–2024)',
        'pop_growth_pct': 'Population Growth (%)',
    },
    color_discrete_sequence=['#2ecc71', '#e67e22', '#e74c3c', '#27ae60'],
)
fig.add_hline(y=0, line_dash='dash', line_color='black')
fig.update_layout(showlegend=False, hovermode='closest')
fig.write_html('Figures/population_vs_affordability_boxplot.html')
fig.show()

### Fig 7 — Top 10 most & least affordable municipalities (animated bar)

In [35]:
years = sorted(df_mun['year'].unique())
frames = []

for year in years:
    df_year = df_mun[df_mun['year'] == year].copy()
    combined = pd.concat([
        df_year.nlargest(10, 'affordability'),
        df_year.nsmallest(10, 'affordability'),
    ]).sort_values('affordability')
    med = combined['affordability'].median()
    frames.append(go.Frame(
        data=[go.Bar(
            x=combined['affordability'],
            y=combined['Municipality'],
            orientation='h',
            marker_color=['#e74c3c' if v < med else '#2ecc71' for v in combined['affordability']],
            text=combined['affordability'].round(1),
            textposition='outside',
        )],
        name=str(year),
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        xaxis=dict(title='Affordability Index', range=[0, df_mun['affordability'].max() * 1.1]),
        yaxis=dict(title=''),
        height=600,
        sliders=[dict(
            active=0,
            currentvalue={'prefix': 'Year: '},
            pad={'t': 50},
            steps=[dict(
                method='animate',
                args=[[str(y)], {'mode': 'immediate', 'frame': {'duration': 300, 'redraw': True},
                                 'transition': {'duration': 200}}],
                label=str(y),
            ) for y in years]
        )],
    ),
)
fig.write_html('Figures/top_bottom_affordability1.html')
fig.show()

### Fig 8 — Multi-layer municipality growth map

In [36]:
MAP_CFG = dict(mapbox_style='carto-positron', mapbox_zoom=5.5,
               mapbox_center={'lat': 56.0, 'lon': 10.5},
               margin={'r': 0, 't': 50, 'l': 0, 'b': 0})

layers = [
    ('pop_growth_pct',       'Pop. Growth (%)',       'Population Growth: %{z:.1f}%'),
    ('aff_change',           'Affordability Change',  'Affordability Change: %{z:.2f}'),
    ('income_growth_pct',    'Income Growth (%)',      'Income Growth: %{z:.1f}%'),
    ('sqm_price_growth_pct', 'sqm Price Growth (%)',  'sqm Price Growth: %{z:.1f}%'),
]

fig = go.Figure()
for i, (col, cb_title, hover_fmt) in enumerate(layers):
    fig.add_trace(go.Choroplethmapbox(
        geojson=geojson_mun,
        locations=df_map['Municipality'],
        featureidkey='properties.name_clean',
        z=df_map[col],
        colorscale='RdYlGn',
        colorbar=dict(title=cb_title, x=1.0),
        hovertemplate=f'<b>%{{location}}</b><br>{hover_fmt}<extra></extra>',
        name=cb_title,
        visible=(i == 0),
    ))

btn_labels = ['Population Growth', 'Affordability Change', 'Income Growth', 'sqm Price Growth']
buttons = [
    dict(label=lbl, method='update',
         args=[{'visible': [j == i for j in range(len(layers))]}])
    for i, lbl in enumerate(btn_labels)
]

fig.update_layout(
    **MAP_CFG,
    updatemenus=[dict(
        type='buttons', direction='left', x=0.5, xanchor='center', y=1.08,
        buttons=buttons, bgcolor='white', bordercolor='grey', font=dict(size=13),
    )],
)
fig.write_html('Figures/full_municipality_growth_map.html', include_plotlyjs='cdn')
fig.show(renderer='browser')

/var/folders/g_/6l1md_mx5c7cn5x0jkgctk0w0000gn/T/ipykernel_58908/2008080931.py:14: DeprecationWarning:

*choroplethmapbox* is deprecated! Use *choroplethmap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



### Fig 9 — Affordability distribution (KDE, selected years)

In [37]:
years_to_plot = [2008, 2015, 2019, 2024]
colors = ['#3498db', '#e67e22', '#9b59b6', '#e74c3c']

data = [df_mun[df_mun['year'] == y]['affordability'].dropna().values for y in years_to_plot]

fig = ff.create_distplot(
    data,
    group_labels=[str(y) for y in years_to_plot],
    colors=colors,
    show_hist=False,
    show_rug=False,
)
fig.update_layout(
    xaxis_title='Affordability Index',
    yaxis_title='Density',
    hovermode='x unified',
)
fig.write_html('Figures/affordability_distribution.html')
fig.show()

### Fig 10 — Income vs sqm price growth scatter (2008 → 2024)

In [38]:
df_scatter = df_mun[df_mun['year'] == 2008][['Municipality', 'Region', 'population']].merge(
    df_growth, on='Municipality'
).dropna()
df_scatter['left_behind'] = df_scatter['sqm_price_growth_pct'] - df_scatter['income_growth_pct']

x_range = [df_scatter['income_growth_pct'].min(), df_scatter['income_growth_pct'].max()]

fig = px.scatter(
    df_scatter,
    x='income_growth_pct', y='sqm_price_growth_pct',
    color='Region', size='population', hover_name='Municipality', size_max=40,
    labels={'income_growth_pct': 'Income Growth (%)', 'sqm_price_growth_pct': 'sqm Price Growth (%)'},
    hover_data={'left_behind': ':.1f'},
)
fig.add_shape(type='line', x0=x_range[0], y0=x_range[0], x1=x_range[1], y1=x_range[1],
              line=dict(color='black', dash='dash', width=1.5))
fig.add_annotation(x=x_range[1] * 0.85, y=x_range[1] * 1.05,
                   text='Prices = Income growth', showarrow=False, font=dict(size=11, color='black'))
fig.update_layout(hovermode='closest')
fig.write_html('Figures/income_vs_price_growth.html')
fig.show()

### Fig 11 — Transactions per capita scatter (2008 vs 2024)

In [41]:
df_s08 = df_mun[df_mun['year'] == 2008][['Municipality', 'Region', 'sales_per_1000', 'population']].rename(
    columns={'sales_per_1000': 'sales_2008'})
df_s24 = df_mun[df_mun['year'] == 2024][['Municipality', 'sales_per_1000']].rename(
    columns={'sales_per_1000': 'sales_2024'})
df_tx = df_s08.merge(df_s24, on='Municipality').dropna()
df_tx['sales_growth_pct'] = (df_tx['sales_2024'] - df_tx['sales_2008']) / df_tx['sales_2008'] * 100

x_range = [df_tx['sales_2008'].min(), df_tx['sales_2008'].max()]

fig = px.scatter(
    df_tx,
    x='sales_2008', y='sales_2024',
    color='Region', size='population', hover_name='Municipality', size_max=40,
    labels={'sales_2008': 'Transactions per 1000 (2008)', 'sales_2024': 'Transactions per 1000 (2024)'},
    hover_data={'sales_growth_pct': ':.1f'},
)
fig.add_shape(type='line', x0=x_range[0], y0=x_range[0], x1=x_range[1], y1=x_range[1],
              line=dict(color='black', dash='dash', width=1.5))
fig.add_annotation(x=x_range[1] * 0.85, y=x_range[1] * 1.05,
                   text='Same transaction intensity', showarrow=False,
                   font=dict(size=11, color='black'))
fig.update_layout(hovermode='closest', template='plotly_white')
fig.write_html('Figures/transactions_per_capita_comparison.html')
fig.show()

### Fig 12 — Normalized regional growth (sqm price / income / population / affordability)

In [42]:
df_reg_s = df_reg.sort_values(['Region', 'year'])

for col, new_col in [
    ('sqm_price_mean', 'sqm_price_norm'),
    ('population',     'population_norm'),
    ('income',         'income_norm'),
    ('affordability',  'affordability_norm'),
]:
    df_reg_s[new_col] = df_reg_s.groupby('Region')[col].transform(lambda x: x / x.iloc[0] * 100)

# Denmark average
df_avg = df_reg_s.groupby('year').agg(
    sqm_price_mean=('sqm_price_mean', 'mean'),
    population=('population', 'mean'),
    income=('income', 'mean'),
    affordability=('affordability', 'mean'),
).reset_index()
df_avg['Region'] = 'Denmark Average'
for col, new_col in [
    ('sqm_price_mean', 'sqm_price_norm'),
    ('population',     'population_norm'),
    ('income',         'income_norm'),
    ('affordability',  'affordability_norm'),
]:
    df_avg[new_col] = df_avg[col] / df_avg[col].iloc[0] * 100

df_all = pd.concat([df_reg_s, df_avg], ignore_index=True)
regions = ['Denmark Average'] + [r for r in df_all['Region'].unique() if r != 'Denmark Average']
traces_per_region = 4

fig = go.Figure()
for i, region in enumerate(regions):
    sub = df_all[df_all['Region'] == region]
    visible = (i == 0)
    for col, name, dash in [
        ('sqm_price_norm',   'sqm Price',    'solid'),
        ('population_norm',  'Population',   'dot'),
        ('income_norm',      'Income',       'dash'),
        ('affordability_norm','Affordability','longdash'),
    ]:
        fig.add_trace(go.Scatter(
            x=sub['year'], y=sub[col], mode='lines',
            name=name, visible=visible, line=dict(dash=dash),
        ))

buttons = []
for i, region in enumerate(regions):
    vis = [False] * (len(regions) * traces_per_region)
    for k in range(traces_per_region):
        vis[i * traces_per_region + k] = True
    buttons.append(dict(
        label=region, method='update',
        args=[{'visible': vis}, {'title': f'Normalized Growth — {region}'}],
    ))

fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction='down', showactive=True)],
    title='Normalized Growth — Denmark Average',
    xaxis_title='Year', yaxis_title='Index (Base = 100)',
    hovermode='x unified',
)
fig.write_html('Figures/normalized_regional_growth.html')
fig.show()